# Simulation 1 Cued Unguided Retrieval Sequence Figure

Figure-only renderer for the Simulation 1 unguided sequence-summary figure. This notebook reads saved example-row and aggregate CSVs; it does not rerun the simulation.


In [1]:
import csv
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch
from jaxcmr.helpers import find_project_root

from selective_interference_v2 import PHASE_COLORS, save_figure

warnings.filterwarnings("ignore")
plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]


In [2]:
# --- figure configuration ---
PROJECT_ROOT = ""
FIGURE_DIR = ""
FIGURE_STR = ""

SEQUENCE_ROWS_PATH = ""
PHASE_TOTALS_PATH = ""
SEQUENCE_SELECTION_PATH = ""

RASTER_MAX_OUTPUTS = 16
RASTER_N_TRIALS = 12

TASK_MCF_VALUES = [1.0, 2.0]
TASK_MCF_LABELS = {
    1.0: "Weak task encoding",
    2.0: "Strong task encoding",
}
REMINDER_CONDITIONS = ["No reminder", "With reminder"]
BAR_REMINDER_LABELS = {
    "No reminder": "No reminder",
    "With reminder": "Film reminder",
}


In [3]:
# Parameters
FIGURE_DIR = "figures"
FIGURE_STR = "simulation1_unguided_cued"


In [4]:
project_root = Path(PROJECT_ROOT).expanduser() if PROJECT_ROOT else find_project_root()
figure_dir = Path(FIGURE_DIR).expanduser() if FIGURE_DIR else None
if figure_dir is not None and not figure_dir.is_absolute():
    figure_dir = project_root / figure_dir

if figure_dir is None or not FIGURE_STR:
    raise ValueError("FIGURE_DIR and FIGURE_STR are required for figure-only rendering.")

sequence_rows_path = Path(SEQUENCE_ROWS_PATH).expanduser() if SEQUENCE_ROWS_PATH else figure_dir / f"{FIGURE_STR}_sequence_rows.csv"
phase_totals_path = Path(PHASE_TOTALS_PATH).expanduser() if PHASE_TOTALS_PATH else figure_dir / f"{FIGURE_STR}_phase_totals_sequence_summary.csv"
sequence_selection_path = Path(SEQUENCE_SELECTION_PATH).expanduser() if SEQUENCE_SELECTION_PATH else figure_dir / f"{FIGURE_STR}_sequence_selection_summary.csv"

print(f"Reading sequence rows from {sequence_rows_path}")
print(f"Reading phase totals from {phase_totals_path}")
print(f"Reading selection summary from {sequence_selection_path}")


Reading sequence rows from /Users/jordangunn/workspace/selective_interference_v2/figures/simulation1_unguided_cued_sequence_rows.csv
Reading phase totals from /Users/jordangunn/workspace/selective_interference_v2/figures/simulation1_unguided_cued_phase_totals_sequence_summary.csv
Reading selection summary from /Users/jordangunn/workspace/selective_interference_v2/figures/simulation1_unguided_cued_sequence_selection_summary.csv


In [5]:
PANEL_LETTER_FONTSIZE = 16
PANEL_TITLE_FONTSIZE = 13
STRUCTURAL_FONTSIZE = 11
SUPPORT_FONTSIZE = 10

SEQUENCE_FIGURE_SUFFIX = "sequence_summary"

RASTER_PHASE_CODES = {
    "none": 0,
    "film": 1,
    "task": 2,
    "delay_filler": 3,
}
RASTER_PHASE_LABELS = {
    "none": "No output",
    "film": "Film",
    "task": "Task",
    "delay_filler": "Delay/filler",
}
RASTER_PHASE_COLORS = {
    "none": "#F7FAFF",
    "film": PHASE_COLORS["film"],
    "task": PHASE_COLORS["task"],
    "delay_filler": "#9CA3AF",
}


def read_rows(path):
    with path.open(newline="") as handle:
        return list(csv.DictReader(handle))


def coerce_sequence_row(row):
    row = dict(row)
    row["task_mcf_scale"] = float(row["task_mcf_scale"])
    row["sampled_trial_index"] = int(row["sampled_trial_index"])
    row["display_trial"] = int(row["display_trial"])
    row["output_position"] = int(row["output_position"])
    row["item_id"] = int(row["item_id"])
    row["phase_code"] = int(row["phase_code"])
    return row


def coerce_total_row(row):
    row = dict(row)
    row["task_mcf_scale"] = float(row["task_mcf_scale"])
    row["recall_probability_mass"] = float(row["recall_probability_mass"])
    return row


def coerce_selection_row(row):
    row = dict(row)
    row["task_mcf_scale"] = float(row["task_mcf_scale"])
    row["mean_film_count"] = float(row["mean_film_count"])
    row["sampled_trial_index"] = int(row["sampled_trial_index"])
    row["display_trial"] = int(row["display_trial"])
    row["selection_rank"] = int(row["selection_rank"])
    row["film_count"] = int(row["film_count"])
    row["task_count"] = int(row["task_count"])
    row["delay_filler_count"] = int(row["delay_filler_count"])
    row["distance_from_mean"] = float(row["distance_from_mean"])
    return row


def cue_total(reminder_condition, task_scale, phase):
    for row in cue_rows:
        if (
            row["reminder_condition"] == reminder_condition
            and np.isclose(row["task_mcf_scale"], task_scale)
            and row["phase"] == phase
        ):
            return row["recall_probability_mass"]
    return np.nan


def build_sequence_samples(rows):
    samples = {}
    for reminder_condition in REMINDER_CONDITIONS:
        for task_scale in TASK_MCF_VALUES:
            condition_rows = [
                row for row in rows
                if row["reminder_condition"] == reminder_condition
                and np.isclose(row["task_mcf_scale"], float(task_scale))
            ]
            display_trials = sorted({row["display_trial"] for row in condition_rows})
            codes = np.zeros((len(display_trials), RASTER_MAX_OUTPUTS), dtype=int)
            display_lookup = {display_trial: idx for idx, display_trial in enumerate(display_trials)}
            for row in condition_rows:
                output_index = row["output_position"] - 1
                if output_index < RASTER_MAX_OUTPUTS:
                    codes[display_lookup[row["display_trial"]], output_index] = row["phase_code"]
            samples[(reminder_condition, float(task_scale))] = {"codes": codes}
    return samples


sequence_rows = [coerce_sequence_row(row) for row in read_rows(sequence_rows_path)]
cue_rows = [coerce_total_row(row) for row in read_rows(phase_totals_path)]
sequence_selection_rows = [coerce_selection_row(row) for row in read_rows(sequence_selection_path)]
sequence_samples = build_sequence_samples(sequence_rows)

for reminder_condition in REMINDER_CONDITIONS:
    for task_scale in TASK_MCF_VALUES:
        key = (reminder_condition, float(task_scale))
        sample = sequence_samples[key]["codes"]
        print(BAR_REMINDER_LABELS[reminder_condition], TASK_MCF_LABELS[float(task_scale)], sample.shape)


No reminder Weak task encoding (12, 16)
No reminder Strong task encoding (12, 16)
Film reminder Weak task encoding (12, 16)
Film reminder Strong task encoding (12, 16)


In [6]:
def plot_sequence_raster(axis, reminder_condition, task_scale, show_xticks=False):
    sample = sequence_samples[(reminder_condition, float(task_scale))]["codes"]
    cmap = ListedColormap([
        RASTER_PHASE_COLORS["none"],
        RASTER_PHASE_COLORS["film"],
        RASTER_PHASE_COLORS["task"],
        RASTER_PHASE_COLORS["delay_filler"],
    ])
    norm = BoundaryNorm(np.arange(-0.5, 4.5, 1.0), cmap.N)
    axis.imshow(sample, cmap=cmap, norm=norm, aspect="auto", interpolation="none")
    axis.set_xlim(-0.5, RASTER_MAX_OUTPUTS - 0.5)
    axis.set_ylim(sample.shape[0] - 0.5, -0.5)
    if show_xticks:
        tick_positions = [0, 7, min(15, RASTER_MAX_OUTPUTS - 1)]
        axis.set_xticks(tick_positions)
        axis.set_xticklabels([str(position + 1) for position in tick_positions])
        axis.tick_params(axis="x", labelsize=SUPPORT_FONTSIZE, length=2.5, pad=2)
    else:
        axis.set_xticks([])
    axis.set_yticks([])
    axis.set_xticks(np.arange(-0.5, RASTER_MAX_OUTPUTS, 1), minor=True)
    axis.set_yticks(np.arange(-0.5, sample.shape[0], 1), minor=True)
    axis.grid(which="minor", color="white", linewidth=0.5)
    axis.tick_params(which="minor", bottom=False, left=False)
    axis.set_facecolor("#FAFBFC")
    for spine in axis.spines.values():
        spine.set_visible(True)
        spine.set_color("#3D4B5C")
        spine.set_linewidth(1.2)


def plot_2x2_phase_recall(axis, phase, title, color, pale_color, show_legend=False, ylim=None):
    x = np.arange(len(TASK_MCF_VALUES))
    width = 0.34
    offsets = [-width / 2, width / 2]
    all_values = []
    for offset, reminder_condition in zip(offsets, REMINDER_CONDITIONS):
        values = [
            cue_total(reminder_condition, float(task_scale), phase)
            for task_scale in TASK_MCF_VALUES
        ]
        all_values.extend(values)
        is_reminder = reminder_condition == "With reminder"
        axis.bar(
            x + offset,
            values,
            width=width,
            color=color if is_reminder else pale_color,
            edgecolor=color,
            linewidth=1.1,
            label=BAR_REMINDER_LABELS[reminder_condition],
            zorder=2,
        )
    axis.set_title(title, fontsize=PANEL_TITLE_FONTSIZE, pad=8)
    axis.set_xticks(x)
    axis.set_xticklabels([TASK_MCF_LABELS[float(value)] for value in TASK_MCF_VALUES])
    axis.set_ylabel("Mean film items recalled", fontsize=STRUCTURAL_FONTSIZE)
    if ylim is None:
        upper = max(all_values) * 1.14 if all_values else 1.0
        upper = max(1.0, np.ceil(upper * 4) / 4)
        axis.set_ylim(0, upper)
    else:
        axis.set_ylim(ylim)
    axis.tick_params(labelsize=SUPPORT_FONTSIZE)
    axis.set_axisbelow(True)
    axis.grid(axis="y", color="#E7EBF0", linewidth=0.45, alpha=0.45, zorder=0)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    if show_legend:
        legend_handles = [
            Patch(facecolor=pale_color, edgecolor=color, linewidth=1.1, label="No reminder"),
            Patch(facecolor=color, edgecolor=color, linewidth=1.1, label="Film reminder"),
        ]
        axis.legend(
            handles=legend_handles,
            frameon=False,
            fontsize=STRUCTURAL_FONTSIZE,
            loc="upper left",
            bbox_to_anchor=(1.02, 1.0),
            borderaxespad=0.0,
        )


def add_panel_label(axis, label, x=-0.16, y=1.12):
    axis.text(
        x,
        y,
        label,
        transform=axis.transAxes,
        fontsize=PANEL_LETTER_FONTSIZE,
        fontweight="bold",
        va="top",
        ha="left",
    )



In [7]:
film_values = [row["recall_probability_mass"] for row in cue_rows if row["phase"] == "film"]
film_bar_ylim = (0, max(1.0, np.ceil(max(film_values) * 1.14 * 4) / 4))

fig = plt.figure(figsize=(10.8, 6.8))
gs = fig.add_gridspec(
    2,
    4,
    height_ratios=[1.15, 0.95],
    hspace=0.55,
    wspace=0.30,
    left=0.18,
    right=0.985,
    top=0.875,
    bottom=0.090,
)

panel_a_gs = gs[0, :].subgridspec(2, 2, hspace=0.20, wspace=0.12)
panel_a_axes = {}
for row_index, task_scale in enumerate(TASK_MCF_VALUES):
    for col_index, reminder_condition in enumerate(REMINDER_CONDITIONS):
        axis = fig.add_subplot(panel_a_gs[row_index, col_index])
        show_xticks = row_index == len(TASK_MCF_VALUES) - 1
        plot_sequence_raster(axis, reminder_condition, float(task_scale), show_xticks=show_xticks)
        if row_index == 0:
            axis.set_title(
                BAR_REMINDER_LABELS[reminder_condition],
                fontsize=STRUCTURAL_FONTSIZE,
                pad=8,
            )
        if col_index == 0:
            axis.set_ylabel(
                TASK_MCF_LABELS[float(task_scale)],
                fontsize=STRUCTURAL_FONTSIZE,
                labelpad=18,
                rotation=0,
                ha="right",
                va="center",
            )
        if row_index == len(TASK_MCF_VALUES) - 1:
            axis.set_xlabel("output position", fontsize=SUPPORT_FONTSIZE, labelpad=5)
        panel_a_axes[(reminder_condition, float(task_scale))] = axis

legend_handles = [
    Patch(facecolor=RASTER_PHASE_COLORS["film"], edgecolor="none", label=RASTER_PHASE_LABELS["film"]),
    Patch(facecolor=RASTER_PHASE_COLORS["task"], edgecolor="none", label=RASTER_PHASE_LABELS["task"]),
    Patch(facecolor=RASTER_PHASE_COLORS["delay_filler"], edgecolor="none", label=RASTER_PHASE_LABELS["delay_filler"]),
    Patch(facecolor=RASTER_PHASE_COLORS["none"], edgecolor="#C7D0DA", label=RASTER_PHASE_LABELS["none"]),
]
panel_a_left = min(axis.get_position().x0 for axis in panel_a_axes.values())
panel_a_right = max(axis.get_position().x1 for axis in panel_a_axes.values())
panel_a_center = (panel_a_left + panel_a_right) / 2
fig.text(
    panel_a_center,
    0.974,
    "Example recall sequences",
    fontsize=PANEL_TITLE_FONTSIZE,
    ha="center",
    va="top",
)
fig.legend(
    handles=legend_handles,
    frameon=False,
    fontsize=SUPPORT_FONTSIZE,
    loc="upper center",
    bbox_to_anchor=(panel_a_center, 0.944),
    ncol=4,
    columnspacing=1.2,
    handlelength=1.3,
)

fig.text(
    panel_a_left - 0.09,
    0.965,
    "A",
    fontsize=PANEL_LETTER_FONTSIZE,
    fontweight="bold",
    ha="left",
    va="top",
)

axis_b = fig.add_subplot(gs[1, 1:3])
plot_2x2_phase_recall(
    axis_b,
    "film",
    "Film output across all trials",
    PHASE_COLORS["film"],
    "#EAF2FF",
    show_legend=True,
    ylim=film_bar_ylim,
)
add_panel_label(axis_b, "B", x=-0.24, y=1.17)

save_figure(str(figure_dir), FIGURE_STR, SEQUENCE_FIGURE_SUFFIX)

